# Murf Falcon text-to-speech on Amazon SageMaker

This notebook deploys the Murf Falcon model package from AWS Marketplace and shows each way to call it:

1. Real-time synthesis (`InvokeEndpoint`): one request in, a complete audio file out.
2. Streaming synthesis (`InvokeEndpointWithResponseStream`): audio starts arriving about 50 ms after the request.
3. Word timestamps.
4. Batch transform: synthesize a folder of requests from Amazon S3.

**Before you start**

- Subscribe to the listing in AWS Marketplace, choose **Continue to configuration**, pick your Region and copy the **Product ARN**.
- Check Service Quotas → Amazon SageMaker: `ml.g6e.xlarge for endpoint usage` must be at least 1, and `ml.g6e.xlarge for transform job usage` for the batch section (it is 0 by default in many accounts).
- Run this from SageMaker Studio or a notebook instance whose execution role has `AmazonSageMakerFullAccess`.

Paste the ARN into the next cell; no other cell needs changes. Delete the endpoint at the end: it is billed while it runs.

The container runs with network isolation: text never leaves your account.

In [ ]:
import base64, json, time
import boto3, sagemaker
from IPython.display import Audio, display

session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
sm = boto3.client("sagemaker", region_name=region)
runtime = boto3.client("sagemaker-runtime", region_name=region)

# Copy the model package ARN for your Region from the listing's "Configure" page.
MODEL_PACKAGE_ARN = "<model-package-arn-for-your-region>"
INSTANCE_TYPE = "ml.g6e.xlarge"   # also supported: ml.g6.xlarge, ml.g5.xlarge and larger sizes
name = f"murf-falcon-{time.strftime('%Y%m%d-%H%M%S')}"

## Deploy a real-time endpoint

Creating the endpoint takes about 10 minutes.

In [ ]:
sm.create_model(
    ModelName=name,
    ExecutionRoleArn=role,
    PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
    EnableNetworkIsolation=True,
)
sm.create_endpoint_config(
    EndpointConfigName=name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": name,
        "InstanceType": INSTANCE_TYPE,
        "InitialInstanceCount": 1,
        "ContainerStartupHealthCheckTimeoutInSeconds": 900,
    }],
)
sm.create_endpoint(EndpointName=name, EndpointConfigName=name)
sm.get_waiter("endpoint_in_service").wait(EndpointName=name, WaiterConfig={"Delay": 30, "MaxAttempts": 60})
print("endpoint", name, "is InService")

## List voices

In [ ]:
resp = runtime.invoke_endpoint(EndpointName=name, ContentType="application/json",
                               Body=json.dumps({"action": "list_voices"}))
voices = json.loads(resp["Body"].read())["voices"]
print(len(voices), "voices, for example:")
for v in voices[:8]:
    print(" ", v)

## Other languages and styles

Many voices speak several languages natively. `supported_locales` lists each language a voice speaks and the styles available in it; send `locale` and `style` to choose.

In [ ]:
multilingual = [v for v in voices if len(v["supported_locales"]) > 1]
print(len(multilingual), "voices speak more than one language, for example:")
for v in multilingual[:5]:
    print(f"  {v['id']}: {v['supported_locales']}")

request = {"text": "ನಮಸ್ಕಾರ, ನಾನು ನಿಮಗೆ ಹೇಗೆ ಸಹಾಯ ಮಾಡಬಹುದು?", "voice_id": "pa-IN-harman", "locale": "kn_IN"}
body = json.loads(runtime.invoke_endpoint(EndpointName=name, ContentType="application/json",
                                          Body=json.dumps(request))["Body"].read())
display(Audio(base64.b64decode(body["audio"])))

## 1. Real-time synthesis

In [ ]:
request = {
    "text": "Hello from Murf Falcon on Amazon SageMaker. This audio was generated inside your own AWS account.",
    "voice_id": "en-US-natalie",
    "format": "wav",
    "sample_rate": 24000,
}
t0 = time.perf_counter()
resp = runtime.invoke_endpoint(EndpointName=name, ContentType="application/json", Body=json.dumps(request))
body = json.loads(resp["Body"].read())
print(f"{body['duration_s']:.2f} s of audio in {time.perf_counter() - t0:.2f} s")
display(Audio(base64.b64decode(body["audio"])))

Set `"response_format": "binary"` to get the audio bytes directly instead of JSON.

## 2. Streaming synthesis

With `"stream": true` the endpoint sends audio as it is generated, so playback can start almost immediately.

In [ ]:
request = {
    "text": "Streaming lets a voice agent start speaking right away, while the rest of the sentence is still being generated.",
    "voice_id": "en-US-natalie",
    "format": "mp3",
    "stream": True,
}
t0 = time.perf_counter()
resp = runtime.invoke_endpoint_with_response_stream(EndpointName=name, ContentType="application/json",
                                                    Body=json.dumps(request))
audio, first = bytearray(), None
for event in resp["Body"]:
    part = event.get("PayloadPart")
    if part:
        first = first or time.perf_counter() - t0
        audio += part["Bytes"]
print(f"first audio after {first * 1000:.0f} ms, finished in {time.perf_counter() - t0:.2f} s")
display(Audio(bytes(audio)))

## 3. Word timestamps

In [ ]:
request = {"text": "Every word comes back with its start and end time.", "voice_id": "en-US-natalie",
           "format": "wav", "add_timestamps": True}
body = json.loads(runtime.invoke_endpoint(EndpointName=name, ContentType="application/json",
                                          Body=json.dumps(request))["Body"].read())
for w in body["words"]:
    print(f"{w['start']:5.2f}-{w['end']:5.2f}  {w['word']}")

## 4. Batch transform

Each JSON file in the input prefix is one request; the output prefix gets one result per file.

In [ ]:
samples = {
    "welcome.json": {"text": "Welcome to our store. How can I help you today?", "voice_id": "en-US-natalie", "format": "mp3", "response_format": "binary"},
    "reminder.json": {"text": "This is a reminder that your appointment is tomorrow at ten AM.", "voice_id": "en-US-natalie", "format": "mp3", "response_format": "binary"},
}
s3 = boto3.client("s3", region_name=region)
prefix = f"murf-falcon-batch/{name}"
for key, req in samples.items():
    s3.put_object(Bucket=bucket, Key=f"{prefix}/input/{key}", Body=json.dumps(req).encode())

sm.create_transform_job(
    TransformJobName=name,
    ModelName=name,
    MaxConcurrentTransforms=4,
    TransformInput={
        "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{bucket}/{prefix}/input/"}},
        "ContentType": "application/json",
        "SplitType": "None",
    },
    TransformOutput={"S3OutputPath": f"s3://{bucket}/{prefix}/output/", "Accept": "audio/mpeg"},
    TransformResources={"InstanceType": INSTANCE_TYPE, "InstanceCount": 1},
)
sm.get_waiter("transform_job_completed_or_stopped").wait(TransformJobName=name, WaiterConfig={"Delay": 30, "MaxAttempts": 60})
print(sm.describe_transform_job(TransformJobName=name)["TransformJobStatus"])
for key in samples:
    audio = s3.get_object(Bucket=bucket, Key=f"{prefix}/output/{key}.out")["Body"].read()
    print(key); display(Audio(audio))

## Clean up

The endpoint is billed while it runs. Delete it when you are done.

In [ ]:
sm.delete_endpoint(EndpointName=name)
sm.delete_endpoint_config(EndpointConfigName=name)
sm.delete_model(ModelName=name)